# Testing a line-number vector plus a phase-dependent 2D component

This notebook tests the decomposition

\[h_{i,t} pprox \mu + v\,k_{i,t} + W z(u_{i,t})\]

where `k` is the current line number, `u = character_count / line_length` is within-line phase, `v` is a single line-number direction, and `W z(u)` is an approximately two-dimensional phase trajectory.

The tests are performed in the full activation space before the final attention layer, using held-out examples. The notebook also evaluates PCA-projected reconstructions inserted back into the model.

In [39]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import SplineTransformer

import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

RUN_DIR = Path("../results/line_breaks_results")
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "final.pt"
N_TRAIN_EXAMPLES = 2_000
N_EVAL_EXAMPLES = 500
COLLECTION_BATCH_SIZE = 128
TRAIN_SEED = 25_001
EVAL_SEED = 25_002
N_SPLINE_KNOTS = 12
RIDGE_ALPHA = 1.0
PREDICTION_MODEL = "phase_plus_line"  # Options listed below.
PREDICTION_MODEL_OPTIONS = {
    "phase": "phase only",
    "phase_plus_line": "phase + one line-number vector",
    "phase_plus_line_onehot": "phase + separate line offsets",
    "phase_line_interaction": "phase + line/phase interaction",
}


In [40]:
with open(RUN_DIR / "config.json", "r") as f:
    cfg = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
vocab = project_tokens.build_vocab(cfg["task"])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(path=CHECKPOINT_PATH, model=model, map_location=device)
model.eval()

final_layer = cfg["model"]["n_layers"] - 1
ACTIVATION_SITE = f"blocks.{final_layer}.hook_resid_pre"
seq_len = cfg["task"]["seq_len"]
print(f"Loaded checkpoint step {checkpoint['step']} on {device}")
print(f"Activation site: {ACTIVATION_SITE}")


Building vocab...
('BOS', '_NEWLINE_')
Loaded checkpoint step 10000 on cpu
Activation site: blocks.2.hook_resid_pre


## Collect activations and line-state variables

In [41]:
def state_variables(token_rows, line_lengths):
    all_counts, all_line_indices = [], []
    for row, width in zip(token_rows.detach().cpu().tolist(), line_lengths.detach().cpu().tolist()):
        running_count = 0
        completed_lines = 0
        row_counts, row_lines = [], []
        for token_id in row:
            row_counts.append(running_count if token_id == vocab.newline_id else running_count + project_data.extract_character_count(vocab.decode_token(token_id)))
            row_lines.append(completed_lines)
            if token_id == vocab.newline_id:
                running_count = 0
                completed_lines += 1
            else:
                running_count += project_data.extract_character_count(vocab.decode_token(token_id))
        all_counts.append(row_counts)
        all_line_indices.append(row_lines)
    return np.asarray(all_counts, dtype=np.float32), np.asarray(all_line_indices, dtype=np.float32)

@torch.no_grad()
def collect_examples(n_examples, seed):
    activation_chunks, count_chunks, line_chunks, width_chunks = [], [], [], []
    collected = 0
    batch_number = 0
    while collected < n_examples:
        size = min(COLLECTION_BATCH_SIZE, n_examples - collected)
        batch = project_data.make_batch(size, vocab, cfg["task"], device,
                                        seed=seed + batch_number * COLLECTION_BATCH_SIZE)
        _, cache = model.run_with_cache(batch.tokens, prepend_bos=False,
                                        names_filter=lambda name: name == ACTIVATION_SITE)
        counts, line_indices = state_variables(batch.tokens, batch.line_lengths)
        activation_chunks.append(cache[ACTIVATION_SITE].detach().cpu())
        count_chunks.append(counts)
        line_chunks.append(line_indices)
        width_chunks.append(batch.line_lengths.detach().cpu().numpy().astype(np.float32))
        collected += size
        batch_number += 1
    return (
        torch.cat(activation_chunks).numpy()[:n_examples],
        np.concatenate(count_chunks)[:n_examples],
        np.concatenate(line_chunks)[:n_examples],
        np.concatenate(width_chunks)[:n_examples],
    )

X_train, counts_train, lines_train, widths_train = collect_examples(N_TRAIN_EXAMPLES, TRAIN_SEED)
X_eval, counts_eval, lines_eval, widths_eval = collect_examples(N_EVAL_EXAMPLES, EVAL_SEED)
d_model = X_train.shape[-1]
print("Train/eval activation shapes:", X_train.shape, X_eval.shape)
print("Observed line-index range:", lines_train.min(), "to", lines_train.max())


Train/eval activation shapes: (2000, 150, 128) (500, 150, 128)
Observed line-index range: 0.0 to 12.0


## Build nested full-space trajectory models

In [42]:
width_mean = widths_train.mean()
width_std = max(widths_train.std(), 1.0)
max_line_index = int(max(lines_train.max(), lines_eval.max()))
phase_spline = SplineTransformer(n_knots=N_SPLINE_KNOTS, degree=3, include_bias=False)
phase_spline.fit(np.linspace(0.0, 1.5, 500)[:, None])
n_phase_features = phase_spline.n_features_out_

def phase_features(counts, widths):
    phase = np.clip(counts / widths[:, None], 0.0, 1.5)
    return phase_spline.transform(phase.reshape(-1, 1))

def design_matrix(counts, lines, widths, mode):
    n_examples, n_positions = counts.shape
    phase = phase_features(counts, widths)
    line_scaled = ((lines - lines_train.mean()) / max(lines_train.std(), 1.0)).reshape(-1, 1)
    phase_line = phase * line_scaled
    if mode == "phase":
        return phase
    if mode == "phase_plus_line":
        return np.concatenate([phase, line_scaled], axis=1)
    if mode == "phase_plus_line_onehot":
        onehot = np.eye(max_line_index + 1, dtype=np.float32)[lines.astype(int).reshape(-1)]
        return np.concatenate([phase, onehot], axis=1)
    if mode == "phase_line_interaction":
        return np.concatenate([phase, phase_line, line_scaled], axis=1)
    raise ValueError(mode)

X_train_flat = X_train.reshape(-1, d_model)
X_eval_flat = X_eval.reshape(-1, d_model)
model_specs = {
    "phase": "phase only",
    "phase_plus_line": "phase + one line-number vector",
    "phase_plus_line_onehot": "phase + separate line offsets",
    "phase_line_interaction": "phase + line/phase interaction",
}
models, predictions = {}, {}
for mode, label in model_specs.items():
    F_train = design_matrix(counts_train, lines_train, widths_train, mode)
    F_eval = design_matrix(counts_eval, lines_eval, widths_eval, mode)
    fitted = Ridge(alpha=RIDGE_ALPHA).fit(F_train, X_train_flat)
    models[mode] = fitted
    predictions[mode] = fitted.predict(F_eval).reshape(N_EVAL_EXAMPLES, seq_len, d_model)
    print(label, "features:", F_train.shape[1])


phase only features: 13
phase + one line-number vector features: 14
phase + separate line offsets features: 26
phase + line/phase interaction features: 27


## Held-out comparison of nested models

In [43]:
rows = []
for mode, label in model_specs.items():
    predicted = predictions[mode]
    rows.append({
        "model": label,
        "MSE": np.mean((X_eval - predicted) ** 2),
        "weighted_R2": r2_score(X_eval_flat, predicted.reshape(-1, d_model), multioutput="variance_weighted"),
    })
mean_baseline = X_train_flat.mean(axis=0)
rows.insert(0, {
    "model": "global mean",
    "MSE": np.mean((X_eval_flat - mean_baseline) ** 2),
    "weighted_R2": r2_score(X_eval_flat, np.broadcast_to(mean_baseline, X_eval_flat.shape), multioutput="variance_weighted"),
})
model_results = pd.DataFrame(rows)
display(model_results.style.format({"MSE": "{:.6f}", "weighted_R2": "{:.4f}"}))


,model,MSE,weighted_R2
0,global mean,4.809923,-0.0001
1,phase only,2.402772,0.5004
2,phase + one line-number vector,1.651311,0.6566
3,phase + separate line offsets,1.165029,0.7578
4,phase + line/phase interaction,1.541549,0.6795


## PCA-projected hierarchy interventions

The hierarchy models above predict full activations. This section projects each prediction into a shared global 3D PCA space and reconstructs it into `d_model` space using the PCA basis. These reconstructed activations are then inserted before layer 2 during evaluation.

Variance reduction is measured relative to replacing every activation with the training-set global mean:

\[1 - \frac{\mathrm{MSE}(h,\hat h)}{\mathrm{MSE}(h,\bar h)}.\]


In [44]:
global_pca = PCA(n_components=3, svd_solver="randomized", random_state=25_003)
global_pca.fit(X_train_flat)

pca_projected_predictions = {}
for mode, full_prediction in predictions.items():
    projected_coords = global_pca.transform(full_prediction.reshape(-1, d_model))
    pca_projected_predictions[mode] = global_pca.inverse_transform(projected_coords).reshape(
        N_EVAL_EXAMPLES, seq_len, d_model
    )

oracle_pca_prediction = global_pca.inverse_transform(
    global_pca.transform(X_eval_flat)
).reshape(N_EVAL_EXAMPLES, seq_len, d_model)

activation_baseline = np.broadcast_to(X_train_flat.mean(axis=0), X_eval_flat.shape)
baseline_mse = float(np.mean((X_eval_flat - activation_baseline) ** 2))
print(f"Global PCA variance explained: {global_pca.explained_variance_ratio_.sum():.1%}")
print(f"Global-mean activation baseline MSE: {baseline_mse:.6f}")


Global PCA variance explained: 88.4%
Global-mean activation baseline MSE: 4.809923


In [45]:
@torch.no_grad()
def evaluate_activation_replacement(replacement_activations):
    correct = total = true_positives = actual_newlines = predicted_newlines = 0
    for start in range(0, N_EVAL_EXAMPLES, COLLECTION_BATCH_SIZE):
        size = min(COLLECTION_BATCH_SIZE, N_EVAL_EXAMPLES - start)
        batch = project_data.make_batch(
            batch_size=size, vocab=vocab, task_cfg=cfg["task"], device=device,
            seed=EVAL_SEED + start,
        )
        replacement = torch.as_tensor(
            replacement_activations[start:start + size],
            device=device,
            dtype=model.W_Q.dtype,
        )
        def replace_activation(activation, hook, replacement=replacement):
            return replacement.to(device=activation.device, dtype=activation.dtype)
        logits = model.run_with_hooks(
            batch.tokens, prepend_bos=False,
            fwd_hooks=[(ACTIVATION_SITE, replace_activation)],
        )
        predictions_here = logits.argmax(dim=-1)
        targets = batch.targets
        target_newline = targets == vocab.newline_id
        prediction_newline = predictions_here == vocab.newline_id
        correct += int((predictions_here == targets).sum().item())
        total += predictions_here.numel()
        true_positives += int((target_newline & prediction_newline).sum().item())
        actual_newlines += int(target_newline.sum().item())
        predicted_newlines += int(prediction_newline.sum().item())
    return {
        "overall_accuracy": correct / total,
        "newline_precision": true_positives / predicted_newlines if predicted_newlines else np.nan,
        "newline_recall": true_positives / actual_newlines if actual_newlines else np.nan,
        "newline_prediction_frequency": predicted_newlines / total,
    }

# Evaluate the clean model on the same held-out examples for reference.
@torch.no_grad()
def evaluate_clean():
    correct = total = true_positives = actual_newlines = predicted_newlines = 0
    for start in range(0, N_EVAL_EXAMPLES, COLLECTION_BATCH_SIZE):
        size = min(COLLECTION_BATCH_SIZE, N_EVAL_EXAMPLES - start)
        batch = project_data.make_batch(
            batch_size=size, vocab=vocab, task_cfg=cfg["task"], device=device,
            seed=EVAL_SEED + start,
        )
        logits = model(batch.tokens)
        predictions_here = logits.argmax(dim=-1)
        targets = batch.targets
        target_newline = targets == vocab.newline_id
        prediction_newline = predictions_here == vocab.newline_id
        correct += int((predictions_here == targets).sum().item())
        total += predictions_here.numel()
        true_positives += int((target_newline & prediction_newline).sum().item())
        actual_newlines += int(target_newline.sum().item())
        predicted_newlines += int(prediction_newline.sum().item())
    return {
        "overall_accuracy": correct / total,
        "newline_precision": true_positives / predicted_newlines if predicted_newlines else np.nan,
        "newline_recall": true_positives / actual_newlines if actual_newlines else np.nan,
        "newline_prediction_frequency": predicted_newlines / total,
    }

activation_replacement_results = []
clean_accuracy = evaluate_clean()
activation_replacement_results.append({"condition": "clean", **clean_accuracy,
    "activation_mse": np.nan, "variance_reduction": np.nan})

replacement_sets = {"oracle PCA reconstruction": oracle_pca_prediction}
replacement_sets.update({model_specs[mode]: value for mode, value in pca_projected_predictions.items()})
for condition, replacement in replacement_sets.items():
    activation_mse = float(np.mean((X_eval - replacement) ** 2))
    metrics = evaluate_activation_replacement(replacement)
    activation_replacement_results.append({
        "condition": condition,
        **metrics,
        "activation_mse": activation_mse,
        "variance_reduction": 1.0 - activation_mse / baseline_mse,
    })

pca_intervention_results = pd.DataFrame(activation_replacement_results)
display(pca_intervention_results.style.format({
    "overall_accuracy": "{:.4f}",
    "newline_precision": "{:.4f}",
    "newline_recall": "{:.4f}",
    "newline_prediction_frequency": "{:.4f}",
    "activation_mse": "{:.6f}",
    "variance_reduction": "{:+.2%}",
}))


,condition,overall_accuracy,newline_precision,newline_recall,newline_prediction_frequency,activation_mse,variance_reduction
0,clean,0.0643,0.5179,0.9997,0.1028,nan,+nan%
1,oracle PCA reconstruction,0.0643,0.4582,0.9927,0.1153,0.558239,+88.39%
2,phase only,0.0628,0.2186,1.0000,0.2436,2.536602,+47.26%
3,phase + one line-number vector,0.0605,0.2193,0.9514,0.2310,1.807950,+62.41%
4,phase + separate line offsets,0.0372,0.1812,0.5026,0.1477,1.363967,+71.64%
5,phase + line/phase interaction,0.0430,0.1873,0.6151,0.1748,1.733915,+63.95%


## Is the line-number effect approximately one-dimensional?

In [46]:
line_model = models["phase_plus_line"]
line_feature_index = n_phase_features
line_vector = line_model.coef_[:, line_feature_index]
line_vector_norm = np.linalg.norm(line_vector)

onehot_model = models["phase_plus_line_onehot"]
line_offset_matrix = onehot_model.coef_[:, n_phase_features:]
line_singular_values = np.linalg.svd(line_offset_matrix.T, compute_uv=False)
line_explained = line_singular_values ** 2 / np.sum(line_singular_values ** 2)

print(f"Single line-number vector norm: {line_vector_norm:.6f}")
print("Line-offset singular-value variance fractions:", line_explained[:6])
print(f"Variance captured by first line-offset direction: {line_explained[0]:.2%}")


Single line-number vector norm: 9.877254
Line-offset singular-value variance fractions: [0.74494594 0.21239021 0.02293754 0.01237982 0.00451295 0.00256136]
Variance captured by first line-offset direction: 74.49%


## Is the phase component effectively two-dimensional and orthogonal?

In [47]:
# Use the phase contribution learned by the additive phase + line model.
phase_coef = line_model.coef_[:, :n_phase_features]
phase_grid = np.linspace(0.0, 1.5, 200)
phase_grid_features = phase_spline.transform(phase_grid[:, None])
phase_curve = phase_grid_features @ phase_coef.T
phase_curve -= phase_curve.mean(axis=0, keepdims=True)
phase_u, phase_singular_values, phase_vt = np.linalg.svd(phase_curve, full_matrices=False)
phase_explained = phase_singular_values ** 2 / np.sum(phase_singular_values ** 2)
phase_basis = phase_vt[:2].T

line_unit = line_vector / max(line_vector_norm, 1e-12)
orthogonality = np.abs(phase_basis.T @ line_unit)
print("Phase singular-value variance fractions:", phase_explained[:6])
print(f"Phase variance captured by first two directions: {phase_explained[:2].sum():.2%}")
print("Absolute cosine with line vector for phase directions:", orthogonality)

phase_coordinates = phase_curve @ phase_basis
phase_radius = np.linalg.norm(phase_coordinates, axis=1)
phase_angle = np.unwrap(np.arctan2(phase_coordinates[:, 1], phase_coordinates[:, 0]))
phase_regression = LinearRegression().fit(phase_grid[:, None], phase_radius)
angle_regression = LinearRegression().fit(phase_grid[:, None], phase_angle)
print(f"Linear radius-vs-phase R²: {phase_regression.score(phase_grid[:, None], phase_radius):.4f}")
print(f"Linear angle-vs-phase R²: {angle_regression.score(phase_grid[:, None], phase_angle):.4f}")


Phase singular-value variance fractions: [8.38040046e-01 7.37041574e-02 6.19950410e-02 1.99795525e-02
 5.72295588e-03 4.14157602e-04]
Phase variance captured by first two directions: 91.17%
Absolute cosine with line vector for phase directions: [0.35832717 0.562101  ]
Linear radius-vs-phase R²: 0.0044
Linear angle-vs-phase R²: 0.6979


## Phase curve in its estimated 2D subspace

In [48]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=phase_coordinates[:, 0], y=phase_coordinates[:, 1], mode="lines+markers",
    marker=dict(size=5, color=phase_grid, colorscale="Viridis", colorbar=dict(title="Phase")),
    line=dict(color="royalblue", width=3),
    text=[f"phase={value:.3f}" for value in phase_grid],
    hovertemplate="%{text}<extra>Phase curve</extra>",
))
fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers", marker=dict(size=8, color="black"), name="Origin"))
fig.update_layout(
    title="Estimated phase trajectory after removing its mean",
    xaxis_title="Phase subspace direction 1", yaxis_title="Phase subspace direction 2",
    height=700,
)
display(fig)


## Overlay line and phase directions on one held-out trajectory

The directions are projected into the shared global 3D PCA coordinates used for the intervention analysis. The held-out activation trajectory is mean-centered by the global PCA mean. Colored rays show the learned line-number direction and the two leading phase-subspace directions.


In [49]:
EXAMPLE_SEED = 25_004
if PREDICTION_MODEL not in PREDICTION_MODEL_OPTIONS:
    raise ValueError(
        f"PREDICTION_MODEL must be one of: {list(PREDICTION_MODEL_OPTIONS)}"
    )
example_rng = np.random.default_rng(EXAMPLE_SEED)
example_index = int(example_rng.integers(N_EVAL_EXAMPLES))

# Project the mean-centered trajectory and learned full-space directions into
# the same global PCA basis.
example_coordinates = global_pca.transform(X_eval[example_index])
predicted_coordinates = global_pca.transform(
    predictions[PREDICTION_MODEL][example_index]
)
line_direction_coordinates = line_vector @ global_pca.components_.T
phase_direction_coordinates = phase_basis.T @ global_pca.components_.T

trajectory_scale = float(np.median(np.linalg.norm(example_coordinates, axis=1)))
trajectory_scale = max(trajectory_scale, 1.0)

def scaled_direction(direction, scale=trajectory_scale):
    norm = np.linalg.norm(direction)
    return direction / max(norm, 1e-12) * scale

line_arrow = scaled_direction(line_direction_coordinates)
phase_arrow_1 = scaled_direction(phase_direction_coordinates[0])
phase_arrow_2 = scaled_direction(phase_direction_coordinates[1])

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode="markers", marker=dict(size=8, color="black"),
    name="Origin", hovertemplate="(0, 0, 0)<extra>Origin</extra>"
))
fig.add_trace(go.Scatter3d(
    x=predicted_coordinates[:, 0], y=predicted_coordinates[:, 1], z=predicted_coordinates[:, 2],
    mode="lines", line=dict(color="darkorange", width=4, dash="dash"),
    name=f"Prediction: {PREDICTION_MODEL_OPTIONS[PREDICTION_MODEL]}",
    text=[f"position {i}" for i in range(seq_len)],
    hovertemplate="%{text}<extra>Predicted activation</extra>",
))
fig.add_trace(go.Scatter3d(
    x=example_coordinates[:, 0], y=example_coordinates[:, 1], z=example_coordinates[:, 2],
    mode="lines+markers", marker=dict(size=5, color=np.arange(seq_len), colorscale="Viridis",
    colorbar=dict(title="Token position")), line=dict(color="rgba(80,80,80,0.7)", width=4),
    name="Mean-centered activation trajectory",
    text=[f"position {i}" for i in range(seq_len)],
    hovertemplate="%{text}<extra>Activation</extra>",
))
for endpoint, name, color in [
    (line_arrow, "Line-number direction", "crimson"),
    (phase_arrow_1, "Phase direction 1", "royalblue"),
    (phase_arrow_2, "Phase direction 2", "seagreen"),
]:
    fig.add_trace(go.Scatter3d(
        x=[0, endpoint[0]], y=[0, endpoint[1]], z=[0, endpoint[2]],
        mode="lines+markers", line=dict(color=color, width=8),
        marker=dict(size=7, color=color), name=name,
        hovertemplate=f"{name}<extra></extra>",
    ))
fig.update_layout(
    title=(f"Held-out example {example_index}: trajectory with line/phase directions<br>"
           f"line length={widths_eval[example_index]:.0f}"),
    scene=dict(xaxis_title="Global PCA 1", yaxis_title="Global PCA 2", zaxis_title="Global PCA 3"),
    height=850, margin=dict(l=0, r=0, b=0, t=100),
)
display(fig)
print(f"Displayed held-out example index: {example_index}")
print(f"Displayed prediction model: {PREDICTION_MODEL_OPTIONS[PREDICTION_MODEL]}")


Displayed held-out example index: 407
Displayed prediction model: phase + one line-number vector


## Held-out residuals after removing the additive line effect

In [50]:
# Compare held-out residuals from the additive model with and without line number.
phase_pred = predictions["phase"]
line_pred = predictions["phase_plus_line"]
phase_mse = np.mean((X_eval - phase_pred) ** 2)
line_mse = np.mean((X_eval - line_pred) ** 2)
print(f"Held-out MSE, phase only: {phase_mse:.6f}")
print(f"Held-out MSE, phase + line vector: {line_mse:.6f}")
print(f"Additional MSE reduction from line vector: {1 - line_mse / phase_mse:.2%}")


Held-out MSE, phase only: 2.402772
Held-out MSE, phase + line vector: 1.651311
Additional MSE reduction from line vector: 31.27%
